## Training the Final Chess Data with DecisionTree

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import confusion_matrix, classification_report

### Loaidng the Data

In [3]:
project_dir = os.path.dirname(os.getcwd())
games = pd.read_csv(os.path.join(project_dir,"data/final_chess.csv"), parse_dates=["Date"])

In [4]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 3255656 entries, 0 to 3255655
Data columns (total 98 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   Site                    str           
 1   Date                    datetime64[us]
 2   White                   str           
 3   Black                   str           
 4   Result                  float64       
 5   WhiteElo                float64       
 6   BlackElo                float64       
 7   ECO                     str           
 8   EloDiff                 float64       
 9   absEloDiff              float64       
 10  AvgElo                  float64       
 11  HigherRatedPlayer       int64         
 12  WhiteGames              int64         
 13  WhiteWin                int64         
 14  WhiteLoss               int64         
 15  WhiteDraw               int64         
 16  BlackGames              int64         
 17  BlackWin                int64         
 18  BlackLoss    

### Preparing for Training

In [5]:
X = games.drop(columns=[
    "White", "Black", "Result", "Site", "ECO",
    # White/Black color counts
    'WasBWinRate', 'WasBLossRate', 'WasBDrawRate', 'BasWWinRate', 'BasWLossRate', 'BasWDrawRate',
    "WasBGames", "WasBWin", "WasBLoss", "WasBDraw",
    "BasWGames", "BasWWin", "BasWLoss", "BasWDraw",
])
y1 = games["Result"].map({1: 1, 0.5: 0, 0:1}) # draw or decisive
y2 = games["Result"].map({1: 2, 0.5: 1, 0: 0}) #sklearn only considers integers

"""
0 = Black win
1 = Draw
2 = White win
"""

'\n0 = Black win\n1 = Draw\n2 = White win\n'

In [6]:
Xtrain = X[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
y1train = y1[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
y2train = y2[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
Xval   = X[(X["Date"].dt.year == 2025)].drop(columns="Date")
y1val   = y1[(X["Date"].dt.year == 2025)].drop(columns="Date")
y2val   = y2[(X["Date"].dt.year == 2025)].drop(columns="Date")
Xtest  = X[X["Date"].dt.year == 2026].drop(columns="Date")
y1test  = y1[X["Date"].dt.year == 2026].drop(columns="Date")
y2test  = y2[X["Date"].dt.year == 2026].drop(columns="Date")

## Decision Tree

In [7]:
from sklearn.tree import DecisionTreeClassifier

One thing I observed is that the model's ability to predict draws improves when it is trained on more recent data. This suggests that the patterns related to draws may change over time.

Based on this observation, I plan to use a two-stage prediction strategy. In **Stage 1**, I will build a model to predict whether a match is **Draw or Decisive**. Then, in **Stage 2**, I will build another model to predict the outcome as **Win or Loss**, using only the matches that are predicted to be decisive in Stage 1.

The idea is to let each model focus on a simpler classification problem, while also taking advantage of the fact that recent data seems to be more useful for predicting draws

#### stage 1

In [253]:
dtc_s1 = DecisionTreeClassifier(ccp_alpha=1e-05, class_weight="balanced", max_depth=20,max_leaf_nodes=1000, min_samples_leaf=168,min_samples_split=654, random_state=67)

In [254]:
dtc_s1.fit(Xtrain, y1train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",654
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",168
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",67
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",1000
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"ccp_alpha ccp_alpha: non-negative float, default=0.0Complexity parameter used for Minimal Cost-Complexity Pruning. Thesubtree with the largest cost complexity that is smaller than``ccp_alpha`` will be chosen. By default, no pruning is performed. See:ref:`minimal_cost_complexity_pruning` for details. See:ref:`sphx_glr_auto_examples_tree_plot_cost_complexity_pruning.py`for an example of such pruning... versionadded:: 0.22",1e-05
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""be

In [255]:
dtc_y1_pred_s1 = dtc_s1.predict(Xval)

In [256]:
confusion_matrix(y1val, dtc_y1_pred_s1)

array([[ 45925,  28036],
       [ 75255, 227530]])

In [257]:
classification_report(y1val, dtc_y1_pred_s1).splitlines()

['              precision    recall  f1-score   support',
 '',
 '           0       0.38      0.62      0.47     73961',
 '           1       0.89      0.75      0.82    302785',
 '',
 '    accuracy                           0.73    376746',
 '   macro avg       0.63      0.69      0.64    376746',
 'weighted avg       0.79      0.73      0.75    376746']

#### stage 2

In [258]:
Xtrain = X[(X["Date"].dt.year >= 2012) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
y1train = y1[(X["Date"].dt.year >= 2012) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
y2train = y2[(X["Date"].dt.year >= 2012) & (X["Date"].dt.year <= 2024)].drop(columns="Date")

In [240]:
decisive_mask = y2train != 1

Xtrain_s2 = Xtrain.loc[decisive_mask]
ytrain_s2 = y2train.loc[decisive_mask]

In [241]:
decisive_val_mask = y2val != 1

Xval_s2 = Xval.loc[decisive_val_mask]
yval_s2 = y2val.loc[decisive_val_mask]

In [ ]:
dtc_s2 = DecisionTreeClassifier(ccp_alpha=1e-05, class_weight="balanced", max_depth=20,max_leaf_nodes=1000, min_samples_leaf=168,min_samples_split=654, random_state=67)

In [243]:
dtc_s2.fit(Xtrain_s2, ytrain_s2)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",654
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",168
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",67
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",1000
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"ccp_alpha ccp_alpha: non-negative float, default=0.0Complexity parameter used for Minimal Cost-Complexity Pruning. Thesubtree with the largest cost complexity that is smaller than``ccp_alpha`` will be chosen. By default, no pruning is performed. See:ref:`minimal_cost_complexity_pruning` for details. See:ref:`sphx_glr_auto_examples_tree_plot_cost_complexity_pruning.py`for an example of such pruning... versionadded:: 0.22",1e-05
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""be

In [244]:
dtc_y1_pred_s2 = dtc_s2.predict(Xval_s2)

In [245]:
confusion_matrix(yval_s2, dtc_y1_pred_s2)

array([[101688,  37060],
       [ 44778, 119259]])

In [246]:
classification_report(yval_s2, dtc_y1_pred_s2).splitlines()

['              precision    recall  f1-score   support',
 '',
 '           0       0.69      0.73      0.71    138748',
 '           2       0.76      0.73      0.74    164037',
 '',
 '    accuracy                           0.73    302785',
 '   macro avg       0.73      0.73      0.73    302785',
 'weighted avg       0.73      0.73      0.73    302785']

#### combining the two stages

In [259]:
pred_stage1 = dtc_s1.predict(Xval)
pred_stage2 = dtc_s2.predict(Xval)

In [260]:
final_pred = np.where(
    pred_stage1 == 0,
    1,              # Draw
    pred_stage2     # Black or White
)

In [261]:
confusion_matrix(y2val, final_pred)

array([[77091, 34430, 27227],
       [14072, 45925, 13964],
       [32570, 40825, 90642]])

In [262]:
classification_report(y2val, final_pred).splitlines()

['              precision    recall  f1-score   support',
 '',
 '           0       0.62      0.56      0.59    138748',
 '           1       0.38      0.62      0.47     73961',
 '           2       0.69      0.55      0.61    164037',
 '',
 '    accuracy                           0.57    376746',
 '   macro avg       0.56      0.58      0.56    376746',
 'weighted avg       0.60      0.57      0.58    376746']

The above method didn't really done a good job at predicting the as i thought, so now let's the check a normal dtc on the whole data.

#### Base Decision tree

In [8]:
Xtrain = X[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
y1train = y1[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
y2train = y2[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
Xval   = X[(X["Date"].dt.year == 2025)].drop(columns="Date")
y1val   = y1[(X["Date"].dt.year == 2025)].drop(columns="Date")
y2val   = y2[(X["Date"].dt.year == 2025)].drop(columns="Date")
Xtest  = X[X["Date"].dt.year == 2026].drop(columns="Date")
y1test  = y1[X["Date"].dt.year == 2026].drop(columns="Date")
y2test  = y2[X["Date"].dt.year == 2026].drop(columns="Date")

In [9]:
dtc = DecisionTreeClassifier(ccp_alpha=1e-06, criterion="gini", class_weight={0: 1, 1: 1.5, 2: 1},
                       max_depth=75, max_leaf_nodes=1000, min_samples_leaf=173,
                       min_samples_split=626, random_state=67)

In [ ]:
#dtc.fit(Xtrain, y2train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",75
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",626
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",173
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",67
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",1000
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.","{0: 1, 1: 1.5, 2: 1}"
,"ccp_alpha ccp_alpha: non-negative float, default=0.0Complexity parameter used for Minimal Cost-Complexity Pruning. Thesubtree with the largest cost complexity that is smaller than``ccp_alpha`` will be chosen. By default, no pruning is performed. See:ref:`minimal_cost_complexity_pruning` for details. See:ref:`sphx_glr_auto_examples_tree_plot_cost_complexity_pruning.py`for an example of such pruning... versionadded:: 0.22",1e-06
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrateg

In [ ]:
#joblib.dump(dtc, os.path.join(project_dir, "results/models/dtc_60_base.joblib"))

['/Users/vallurileelasaikrishna/chess/results/models/dtc_60_base.joblib']

In [17]:
model_path = os.path.join(project_dir, "results/models/dtc_60_base.joblib")
dtc = joblib.load(model_path)

In [18]:
y_pred = dtc.predict(Xval)

In [19]:
confusion_matrix(y2val, y_pred)

array([[ 86833,  16843,  35072],
       [ 20437,  29456,  24068],
       [ 33887,  18972, 111178]])

In [20]:
classification_report(y2val, y_pred).splitlines()

['              precision    recall  f1-score   support',
 '',
 '           0       0.62      0.63      0.62    138748',
 '           1       0.45      0.40      0.42     73961',
 '           2       0.65      0.68      0.67    164037',
 '',
 '    accuracy                           0.60    376746',
 '   macro avg       0.57      0.57      0.57    376746',
 'weighted avg       0.60      0.60      0.60    376746']